# Phase 2 — Dataset Consolidation (train / validation / test)

### Why three splits, not two
Phase 4 will tune SNN hyperparameters with Bayesian Optimization. If that tuning is scored against
the same set used for final reporting, the reported number stops being an honest estimate — the
hyperparameters get quietly fit to the test set. So:

- **Train** — real rows + disclosed SMOTE-synthetic rows, used to fit model weights.
- **Validation** — 100% real, held-out dates, used only to score hyperparameter candidates in Phase 4.
- **Test** — 100% real, held-out dates, touched exactly once, at the very end, for the number that
  goes in the results section.

Validation and test dates are disjoint from each other and from training dates. Nothing synthetic
ever enters validation or test.

### On disclosure
The `is_synthetic` column stays in the training file. This is not optional — it's what makes the
pipeline auditable. It only needs to be mentioned once in your methodology text, not repeated in
every figure.


In [ ]:
from google.colab import files

uploaded = files.upload()

## 1. Config

In [ ]:
INPUT_CSV                   = '/content/algae_prototype_dataset_10k.csv'   # your 10k-row real dataset
RANDOM_SEED                 = 42

MIN_PER_CLASS_FOR_HELD_OUT_DATE = 20   # a date must have at least this many of BOTH classes to be eligible for val/test
N_VAL_DATES                 = 1
N_TEST_DATES                = 1

TARGET_TRAIN_PER_CLASS      = 4500     # SMOTE target per class in training
SMOTE_K_NEIGHBORS           = 5

OUTPUT_DIR                  = '/content'
TRAIN_OUT_PATH = f'{OUTPUT_DIR}/algae_dataset_train.csv'
VAL_OUT_PATH   = f'{OUTPUT_DIR}/algae_dataset_val.csv'
TEST_OUT_PATH  = f'{OUTPUT_DIR}/algae_dataset_test.csv'
README_PATH    = f'{OUTPUT_DIR}/DATASET_README.md'

## 2. Install & load

In [ ]:
!pip install imbalanced-learn -q

import numpy as np
import pandas as pd
from datetime import datetime
from imblearn.over_sampling import SMOTE

df = pd.read_csv(INPUT_CSV)
non_feature_cols = ['longitude', 'latitude', 'cyan_dn', 'label', 'source_date']
feature_columns = [c for c in df.columns if c not in non_feature_cols]
assert len(feature_columns) == 31

print('Loaded:', df.shape)
print('Class balance:', df['label'].value_counts().to_dict())
print('Distinct real dates:', sorted(df['source_date'].unique()))

## 3. Pick disjoint validation and test dates (real, class-balanced enough to be usable)
Test matters more than validation here — it's the number that goes in your results section.
Eligible dates are ranked by their minority-class count and the **strongest date goes to test**,
not picked at random; a small, lopsided test set (e.g. 23 real no-bloom examples) would undermine
the one metric you actually need to trust. Validation gets the next-best remaining date.

In [ ]:
date_stats = df.groupby(['source_date', 'label']).size().unstack(fill_value=0)
print('Per-date class counts:')
print(date_stats)

eligible = date_stats[
    (date_stats.get(0, 0) >= MIN_PER_CLASS_FOR_HELD_OUT_DATE) &
    (date_stats.get(1, 0) >= MIN_PER_CLASS_FOR_HELD_OUT_DATE)
].copy()
eligible['min_class_count'] = eligible.min(axis=1)
eligible = eligible.sort_values('min_class_count', ascending=False)
print('\nEligible dates, ranked by minority-class strength:')
print(eligible)

eligible_dates = eligible.index.tolist()
needed = N_VAL_DATES + N_TEST_DATES
assert len(eligible_dates) >= needed, (
    f'Only {len(eligible_dates)} eligible dates but need {needed} for val+test. '
    'Add more real dates via Stage 1, or lower MIN_PER_CLASS_FOR_HELD_OUT_DATE.'
)

# Strongest date(s) -> test (the metric that gets reported); next-best -> validation (tuning only)
test_dates = [str(d) for d in eligible_dates[:N_TEST_DATES]]
val_dates  = [str(d) for d in eligible_dates[N_TEST_DATES:N_TEST_DATES + N_VAL_DATES]]

print('\nTest dates (strongest, used for final metrics):', test_dates)
print('Validation dates (next-best, used for tuning only):', val_dates)

train_pool_df = df[~df['source_date'].isin(val_dates + test_dates)].copy()
val_df  = df[df['source_date'].isin(val_dates)].copy()
test_df = df[df['source_date'].isin(test_dates)].copy()

for name, d in [('Train pool (real, pre-SMOTE)', train_pool_df), ('Validation', val_df), ('Test', test_df)]:
    print(f'\n{name}: {len(d)} rows, classes: {d["label"].value_counts().to_dict()}')
    assert d['label'].nunique() == 2, f'{name} ended up single-class -- adjust date selection.'

## 4. Duplicate check on the real pool (integrity check, not augmentation)

In [ ]:
before = len(train_pool_df) + len(val_df) + len(test_df)
combined_real = pd.concat([train_pool_df, val_df, test_df], ignore_index=True)
n_dupes = combined_real.duplicated(subset=feature_columns).sum()
print('Duplicate feature rows across the full real pool:', n_dupes)
assert n_dupes == 0, 'Unexpected duplicate real pixels across dates -- investigate Stage 1 output before proceeding.'

## 5. SMOTE — training pool only

In [ ]:
X_train_real = train_pool_df[feature_columns].values
y_train_real = train_pool_df['label'].values

current_counts = pd.Series(y_train_real).value_counts().to_dict()
print('Real training class counts before SMOTE:', current_counts)

k = min(SMOTE_K_NEIGHBORS, min(current_counts.values()) - 1)
assert k >= 1, 'Too few real training rows in the minority class to run SMOTE -- add more real dates.'

target_counts = {0: TARGET_TRAIN_PER_CLASS, 1: TARGET_TRAIN_PER_CLASS}
for cls, cnt in current_counts.items():
    if cnt >= target_counts[cls]:
        target_counts[cls] = cnt   # never downsample real data here

smote = SMOTE(sampling_strategy=target_counts, random_state=RANDOM_SEED, k_neighbors=k)
X_res, y_res = smote.fit_resample(X_train_real, y_train_real)

print('Training class counts after SMOTE:', pd.Series(y_res).value_counts().to_dict())
print('Total training rows after SMOTE:', len(X_res))

## 6. Rebuild the training dataframe with honest provenance flags

In [ ]:
n_real = len(X_train_real)

train_final = pd.DataFrame(X_res, columns=feature_columns)
train_final['label'] = y_res
train_final['is_synthetic'] = False
train_final.loc[n_real:, 'is_synthetic'] = True
train_final['source_date'] = 'REAL'
train_final.loc[n_real:, 'source_date'] = 'SYNTHETIC'

for col in ['longitude', 'latitude', 'cyan_dn']:
    train_final[col] = np.nan
    train_final.loc[:n_real-1, col] = train_pool_df[col].values

n_synth = train_final['is_synthetic'].sum()
print(f'Final training set: {len(train_final)} rows '
      f'({n_real} real, {n_synth} synthetic, {n_synth/len(train_final)*100:.1f}% synthetic)')

val_df['is_synthetic'] = False
test_df['is_synthetic'] = False

## 7. Sanity checks

In [ ]:
for name, d in [('Train', train_final), ('Validation', val_df), ('Test', test_df)]:
    n_nan = d[feature_columns].isna().sum().sum()
    n_inf = np.isinf(d[feature_columns].to_numpy()).sum()
    print(f'{name}: NaN={n_nan}  Inf={n_inf}  rows={len(d)}  classes={d["label"].value_counts().to_dict()}')
    assert n_nan == 0 and n_inf == 0

assert not val_df['is_synthetic'].any() and not test_df['is_synthetic'].any()
print('\nConfirmed: validation and test sets contain zero synthetic rows.')
print(f'Combined total (train + val + test): {len(train_final) + len(val_df) + len(test_df)}')

## 8. Save datasets + a README documenting exactly what's in them

In [ ]:
train_final.to_csv(TRAIN_OUT_PATH, index=False)
val_df.to_csv(VAL_OUT_PATH, index=False)
test_df.to_csv(TEST_OUT_PATH, index=False)

readme = f"""# Dataset Composition — CyanoHAB Lake Erie Detection

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}

## Source
Sentinel-2 (31 spectral/spatial features) paired with NOAA CyAN CI_cyano daily GeoTIFFs,
thresholded to a binary label (DN=0 -> no bloom, DN 132-253 -> bloom alert; DN 1-131/254/255 discarded).

## Real data collected
{len(df)} real, labeled pixels across {df['source_date'].nunique()} real satellite acquisition dates:
{sorted(df['source_date'].unique())}

## Splits (by date, disjoint -- no date appears in more than one split)

### Train: {len(train_final)} rows
- Real: {n_real} ({n_real/len(train_final)*100:.1f}%)
- Synthetic (SMOTE): {n_synth} ({n_synth/len(train_final)*100:.1f}%)
- Class balance: {train_final['label'].value_counts().to_dict()}
- Synthetic rows generated via SMOTE (imbalanced-learn), which interpolates between real feature
  vectors and their same-class nearest neighbors. Flagged in the `is_synthetic` column.

### Validation: {len(val_df)} rows -- 100% real, dates: {val_dates}
- Class balance: {val_df['label'].value_counts().to_dict()}
- Used only for hyperparameter tuning (Phase 4), never for training weights or final reporting.

### Test: {len(test_df)} rows -- 100% real, dates: {test_dates}
- Class balance: {test_df['label'].value_counts().to_dict()}
- Held out entirely. Used exactly once, for final reported metrics.

## Disclosure statement (for methodology section)
"The training set was augmented using SMOTE (Synthetic Minority Oversampling Technique), which
generates synthetic samples by interpolating between real feature vectors and their same-class
nearest neighbors. All synthetic samples are confined to the training set and flagged via an
`is_synthetic` column; model evaluation is performed exclusively on real, unaugmented
satellite-derived observations from dates never seen during training or hyperparameter tuning."
"""

with open(README_PATH, 'w') as f:
    f.write(readme)

print('Saved:')
print(' ', TRAIN_OUT_PATH)
print(' ', VAL_OUT_PATH)
print(' ', TEST_OUT_PATH)
print(' ', README_PATH)

In [ ]:
from google.colab import files

files.download('/content/algae_dataset_train.csv')
files.download('/content/algae_dataset_val.csv')
files.download('/content/algae_dataset_test.csv')
files.download('/content/DATASET_README.md')

## Phase 2 complete
Next: Phase 4 (Bayesian-optimized SNN training) uses `algae_dataset_train.csv` for weights and
`algae_dataset_val.csv` to score hyperparameter candidates — `algae_dataset_test.csv` stays
untouched until Phase 5's final evaluation.
